In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [15]:
def setup_driver():
  """Setup Chrome driver with options"""
  chrome_options = Options()
  #chrome_options.add_argument("--headless")

  chrome_options.page_load_strategy = "eager"

  service = Service(ChromeDriverManager().install())
  driver = webdriver.Chrome(service=service, options=chrome_options)
  driver.set_page_load_timeout(60)
  #driver.delete_all_cookies()
  #driver.refresh()
  return driver

In [5]:
url = "https://www.myneta.info/LokSabha2019/"
ua = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"}
html = requests.get(url, headers=ua, timeout=30)
print(html.status_code)

200


States - Andhra Pradesh, Bihar, Gujarat

In [7]:
html = html.text
soup = bs(html, "html.parser")

In [9]:
for card in soup.find_all("div", class_="w3-dropdown-click w3-block"):
    print(card.find("button").get_text(strip=True))
    break

ANDAMAN & NICOBAR ISLANDS


In [10]:
states = ["ANDHRA PRADESH", "BIHAR", "GUJARAT"]
url_dict = {}

In [11]:
found = 0
for card in soup.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict)

{'ANDHRA PRADESH': 'index.php?action=show_constituencies&state_id=34', 'BIHAR': 'index.php?action=show_constituencies&state_id=37', 'GUJARAT': 'index.php?action=show_constituencies&state_id=39'}


In [22]:
data = {}
for state in states:
    data[state] = []

In [24]:
driver = setup_driver()
try:
    for state in states:
        state_url = url+url_dict[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    i+=1
                data[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the ne

In [25]:
print(len(data["ANDHRA PRADESH"]))
print(len(data["BIHAR"]))
print(len(data["GUJARAT"]))

334
619
371


In [26]:
andhra = pd.DataFrame(data["ANDHRA PRADESH"])
bihar = pd.DataFrame(data["BIHAR"])
gujarat = pd.DataFrame(data["GUJARAT"])

In [27]:
cols_names = {
    0: "Sno.",
    1: "Candidate",
    2: "Party",
    3: "Criminal Cases",
    4: "Education",
    5: "Age",
    6: "Total Assets",
    7: "Liabilities"
}

In [28]:
andhra = andhra.rename(columns=cols_names)

In [29]:
bihar = bihar.rename(columns=cols_names)

In [30]:
gujarat = gujarat.rename(columns=cols_names)

In [ ]:
andhra

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
20,1,A. Hemanth,IND,0,5th Pass,28,"Rs 50,000\n~ 50 Thou+",Rs 0\n~
21,2,Abdul Waris,SDPI,4,8th Pass,58,"Rs 51,10,000\n~ 51 Lacs+",Rs 0\n~
22,3,Adala Prabhakara Reddy,YSRCP,0,12th Pass,70,"Rs 2,21,16,63,364\n~ 221 Crore+","Rs 15,22,81,255\n~ 15 Crore+"
23,4,Adari Anand Kumar,TDP,0,10th Pass,43,"Rs 41,05,47,212\n~ 41 Crore+","Rs 20,55,91,400\n~ 20 Crore+"
24,5,Adinarayana Inty,BJP,1,Post Graduate,67,"Rs 3,38,85,898\n~ 3 Crore+","Rs 44,79,155\n~ 44 Lacs+"
...,...,...,...,...,...,...,...,...
329,310,Yanamadala Venkata Suresh,IND,0,Graduate,50,"Rs 13,02,000\n~ 13 Lacs+",Rs 0\n~
330,311,Yarlagadda Rama Mohana Rao,Bahujan Maha Party,0,12th Pass,56,"Rs 1,00,86,000\n~ 1 Crore+",Rs 0\n~
331,312,Yeduguri Sandinti Avinash Reddy,YSRCP,4,Post Graduate,34,"Rs 18,69,69,794\n~ 18 Crore+","Rs 2,28,33,319\n~ 2 Crore+"
332,313,Yella Rao Siyyadula,IND,0,12th Pass,37,"Rs 57,000\n~ 57 Thou+",Rs 0\n~


In [35]:
bihar

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,A M Izharul Haque,Sathi Aur Aapka Faisala Party,0,Post Graduate,43,"Rs 8,54,501\n~ 8 Lacs+",Rs 0\n~
1,2,Aakash Kumar,Rashtriya Lok Samta Party,0,12th Pass,27,"Rs 2,58,67,226\n~ 2 Crore+",Rs 0\n~
2,3,Abdul Ajij,IND,2,8th Pass,69,"Rs 4,35,000\n~ 4 Lacs+",Rs 0\n~
3,4,Abdul Bari Siddiqui,RJD,0,12th Pass,67,"Rs 2,11,27,716\n~ 2 Crore+","Rs 9,11,152\n~ 9 Lacs+"
4,5,Abdul Wahid Khan,IND,1,10th Pass,69,"Rs 15,45,000\n~ 15 Lacs+","Rs 1,19,028\n~ 1 Lacs+"
...,...,...,...,...,...,...,...,...
614,615,Vishnu Dev,IND,0,Graduate,59,"Rs 1,42,10,591\n~ 1 Crore+","Rs 5,00,000\n~ 5 Lacs+"
615,616,Vishnu Dev Yadav,BSP,0,Graduate,67,"Rs 4,34,39,000\n~ 4 Crore+",Rs 0\n~
616,617,Vishnu Priya,Bahujan Mukti Party,0,Graduate,32,"Rs 88,10,000\n~ 88 Lacs+",Rs 0\n~
617,618,Wakil Ahmad,Janata Party,0,Graduate,46,,


In [36]:
gujarat

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,Parmar Vijay,BSP,4,12th Pass,48,"Rs 5,35,000\n~ 5 Lacs+",Rs 0\n~
1,2,A. J. Patel,INC,0,Graduate Professional,73,"Rs 69,87,73,712\n~ 69 Crore+","Rs 9,62,20,673\n~ 9 Crore+"
2,3,Aamin Mamadbhai Safiya,IND,0,8th Pass,30,"Rs 46,100\n~ 46 Thou+",Rs 0\n~
3,4,Aantroliya Karabhai,IND,0,8th Pass,40,"Rs 7,60,000\n~ 7 Lacs+","Rs 13,29,600\n~ 13 Lacs+"
4,5,Abdulkudus Abdulmajid Molapiya,IND,0,Literate,44,"Rs 7,56,000\n~ 7 Lacs+",Rs 0\n~
...,...,...,...,...,...,...,...,...
366,367,Vora Alimahmad Rajabhai,IND,0,Illiterate,67,"Rs 19,70,884\n~ 19 Lacs+","Rs 32,000\n~ 32 Thou+"
367,368,Yasinali Polra,New All India Congress Party,0,12th Pass,47,"Rs 33,63,000\n~ 33 Lacs+","Rs 20,14,250\n~ 20 Lacs+"
368,369,Zala Dalpatsinh Motisinh,IND,0,8th Pass,55,,
369,370,Zala Mukeshbhai Bharmalbhai,IND,1,10th Pass,36,"Rs 1,05,72,716\n~ 1 Crore+","Rs 18,78,780\n~ 18 Lacs+"


In [ ]:
# andhra.to_csv("andhra.csv", index=False)
# bihar.to_csv("bihar.csv", index=False)
# gujarat.to_csv("gujarat.csv", index=False)